In [1]:
import numpy as np
import pandas as pd
from load import *
from aligner import *


[*********************100%***********************]  5 of 5 completed


# 2. Validate

In [2]:
# Missing values
data.isna().sum()

Price   Ticker
Close   GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
High    GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Low     GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Open    GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Volume  GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
dtype: int64

In [3]:
data.index.duplicated().sum()
print(data.shape)
print(data.head())
print(data.columns)

(6539, 25)
Price      Close                               High                            \
Ticker       GLD        QQQ        SPY TLT VNQ  GLD        QQQ        SPY TLT   
Date                                                                            
2000-01-03   NaN  79.839714  91.132759 NaN NaN  NaN  81.051003  92.895103 NaN   
2000-01-04   NaN  74.362549  87.568909 NaN NaN  NaN  78.786383  90.271169 NaN   
2000-01-05   NaN  72.466629  87.725525 NaN NaN  NaN  75.521182  88.685023 NaN   
2000-01-06   NaN  67.489807  86.315689 NaN NaN  NaN  74.151891  88.665481 NaN   
2000-01-07   NaN  75.837158  91.328545 NaN NaN  NaN  75.837158  91.328545 NaN   

Price           ... Open                               Volume            \
Ticker     VNQ  ...  GLD        QQQ        SPY TLT VNQ    GLD       QQQ   
Date            ...                                                       
2000-01-03 NaN  ...  NaN  81.051003  92.895103 NaN NaN    NaN  36345200   
2000-01-04 NaN  ...  NaN  77.522431  89.

In [4]:
close_prices = data["Close"]

for ticker in close_prices.columns:
    print(
        ticker,
        close_prices[ticker].first_valid_index(),
        close_prices[ticker].last_valid_index()
    )

GLD 2004-11-18 00:00:00 2025-12-31 00:00:00
QQQ 2000-01-03 00:00:00 2025-12-31 00:00:00
SPY 2000-01-03 00:00:00 2025-12-31 00:00:00
TLT 2002-07-30 00:00:00 2025-12-31 00:00:00
VNQ 2004-09-29 00:00:00 2025-12-31 00:00:00


# 3. Align assets

In [5]:
# We need to align so all 4 ETF:s start at the same time to match future matrix
# For the moment, we work with only one column
close_prices = data["Close"]

aligned_prices = align_assets(close_prices)
print(aligned_prices.head())
print(aligned_prices.shape)
aligned_prices.isna().sum().sum()

Ticker            GLD        QQQ        SPY        TLT        VNQ
Date                                                             
2004-11-18  44.380001  33.120083  79.745270  43.988667  21.189312
2004-11-19  44.779999  32.605873  78.858757  43.637634  20.970301
2004-11-22  44.950001  32.917747  79.234833  43.865101  21.115015
2004-11-23  44.750000  32.867203  79.355736  43.919441  21.204952
2004-11-24  45.049999  33.153786  79.543793  43.919441  21.572578
(5313, 5)


np.int64(0)

# 4. Convert raw to log returns

In [6]:
# Using r_t = log(P_t / P_{t-1})
log_returns = np.log(aligned_prices / aligned_prices.shift(1))

In [7]:
# Remove the first NaN raw
log_returns = log_returns.dropna()
print(log_returns.shape)
log_returns.describe()

(5312, 5)


Ticker,GLD,QQQ,SPY,TLT,VNQ
count,5312.000000,5312.000000,5312.000000,5312.000000,5312.000000
mean,0.000412,0.000549,0.000403,0.000125,0.000265
std,0.011132,0.013615,0.011983,0.009230,0.018122
min,-0.091905,-0.127592,-0.115886,-0.069010,-0.217083
25%,-0.005113,-0.005178,-0.003944,-0.005346,-0.006470
50%,0.000588,0.001146,0.000718,0.000445,0.000768
75%,0.006217,0.007321,0.005776,0.005505,0.007510
max,0.106974,0.114799,0.135577,0.072502,0.157060


# 5. Create rolling observation window

In [8]:
# Seeing only today's return is not relevant enough
# We also care to see how turbulent has the asset been recently
# we compute the std_dev or returns σ_t=std(r_t−19,...,r_t), 20 is approx. 1 month of trades
volatility_20 = log_returns.rolling(20).std()
print(volatility_20.head(25)) # volatility = 0.009785 ==> 0.97%

Ticker           GLD       QQQ       SPY       TLT       VNQ
Date                                                        
2004-11-19       NaN       NaN       NaN       NaN       NaN
2004-11-22       NaN       NaN       NaN       NaN       NaN
2004-11-23       NaN       NaN       NaN       NaN       NaN
2004-11-24       NaN       NaN       NaN       NaN       NaN
2004-11-26       NaN       NaN       NaN       NaN       NaN
2004-11-29       NaN       NaN       NaN       NaN       NaN
2004-11-30       NaN       NaN       NaN       NaN       NaN
2004-12-01       NaN       NaN       NaN       NaN       NaN
2004-12-02       NaN       NaN       NaN       NaN       NaN
2004-12-03       NaN       NaN       NaN       NaN       NaN
2004-12-06       NaN       NaN       NaN       NaN       NaN
2004-12-07       NaN       NaN       NaN       NaN       NaN
2004-12-08       NaN       NaN       NaN       NaN       NaN
2004-12-09       NaN       NaN       NaN       NaN       NaN
2004-12-10       NaN    

In [9]:
# We add momentum to observe the general direction, up or down
# (momentum is supposed to be smaller than volatility, because avg_return << std_dev)
momentum_20 = log_returns.rolling(20).mean()
print(momentum_20.head(25))

Ticker           GLD       QQQ       SPY       TLT       VNQ
Date                                                        
2004-11-19       NaN       NaN       NaN       NaN       NaN
2004-11-22       NaN       NaN       NaN       NaN       NaN
2004-11-23       NaN       NaN       NaN       NaN       NaN
2004-11-24       NaN       NaN       NaN       NaN       NaN
2004-11-26       NaN       NaN       NaN       NaN       NaN
2004-11-29       NaN       NaN       NaN       NaN       NaN
2004-11-30       NaN       NaN       NaN       NaN       NaN
2004-12-01       NaN       NaN       NaN       NaN       NaN
2004-12-02       NaN       NaN       NaN       NaN       NaN
2004-12-03       NaN       NaN       NaN       NaN       NaN
2004-12-06       NaN       NaN       NaN       NaN       NaN
2004-12-07       NaN       NaN       NaN       NaN       NaN
2004-12-08       NaN       NaN       NaN       NaN       NaN
2004-12-09       NaN       NaN       NaN       NaN       NaN
2004-12-10       NaN    

# 6. Generate derived features

In [13]:
# Combine the features
features = pd.concat(
    [
        log_returns,
        volatility_20,
        momentum_20
    ],
    axis=1,
    keys=["ret", "vol", "mom"]
)
# Drop the first 19 NaN
features = features.dropna()

print(features.shape)
print(features.head())

(5293, 15)
                 ret                                               vol  \
Ticker           GLD       QQQ       SPY       TLT       VNQ       GLD   
Date                                                                     
2004-12-17  0.011608 -0.002555 -0.006692 -0.001351  0.011262  0.009785   
2004-12-20  0.003389 -0.006865  0.000251  0.001801 -0.003506  0.009587   
2004-12-21 -0.002710  0.012171  0.007671  0.003033  0.011349  0.009544   
2004-12-22 -0.004533  0.001260  0.002406 -0.003371  0.007429  0.009546   
2004-12-23  0.005663  0.000755  0.000745 -0.003268 -0.007606  0.009506   

                                                         mom            \
Ticker           QQQ       SPY       TLT       VNQ       GLD       QQQ   
Date                                                                     
2004-12-17  0.008715  0.005472  0.007788  0.008975 -0.000215  0.000705   
2004-12-20  0.008043  0.004732  0.007557  0.008570 -0.000494  0.001144   
2004-12-21  0.008209  0.00

In [11]:
# 7. Normalize

# 8. Create Splits
